#  Import required packages


In [2]:
import pandas as pd
from pathlib import Path
import numpy as np
from natsort import natsorted
import os
import yaml
%matplotlib inline
%config InlineBackend.figure_format='retina'

# Required Inputs


In [4]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Set up directories
raw_dir = Path(config['raw_dir'])
metadata_dir = Path(config['metadata_dir'])
preprocessed_dir = Path(config['preprocessed_dir'])

# make a roi list of directories in raw_dir that are not .DS_Store
roi_list = [roi for roi in raw_dir.iterdir() if roi.is_dir() and roi.name != '.DS_Store']

#make antibody panel list
ab_panels = [ab for ab in metadata_dir.iterdir() if ab.is_file() and 'Ab_panel.xlsx' in ab.name]

# Select the phenotypes depth to take from the MACSIMA output
pheno_depth=int(3)

# Preparing the cell phenotypes dataframe

## 1) Extract phenotype lineage information

In [19]:
 lineage_info={}
 
 lineage_matrix = pd.read_csv(metadata_dir/"expression_table_sophia.csv", sep=';')
 
 for cell_type,lineage in zip(lineage_matrix.iloc[:,0].values,lineage_matrix.iloc[:,1].values):
 
     lineage_info[cell_type]=tuple(np.array(lineage.split('_')).astype('int'))
 
lineage_info

{'T_cells': (np.int64(1), np.int64(0), np.int64(1)),
 'CD4+_T_cells': (np.int64(2), np.int64(1), np.int64(2)),
 'CD8+_T_cells': (np.int64(2), np.int64(1), np.int64(3)),
 'Treg_T_cells': (np.int64(3), np.int64(2), np.int64(4)),
 'B_cells': (np.int64(1), np.int64(0), np.int64(5)),
 'Plasma_cells': (np.int64(2), np.int64(5), np.int64(6)),
 'Granulocytes': (np.int64(1), np.int64(0), np.int64(7)),
 'NK_cells': (np.int64(1), np.int64(0), np.int64(8)),
 'Mast_cells': (np.int64(1), np.int64(0), np.int64(9)),
 'Macrophages': (np.int64(1), np.int64(0), np.int64(10)),
 'M1_macrophages': (np.int64(2), np.int64(10), np.int64(11)),
 'M2_macrophages': (np.int64(2), np.int64(10), np.int64(12)),
 'M1_M2_macrophages': (np.int64(2), np.int64(10), np.int64(13)),
 'Tumor_cells': (np.int64(1), np.int64(0), np.int64(14)),
 'Endothelial_cells': (np.int64(1), np.int64(0), np.int64(15)),
 'Lymphatic_endothelial_cells': (np.int64(1), np.int64(0), np.int64(16)),
 'Actin+_cells': (np.int64(1), np.int64(0), np.int6

## 2) Concatenate all cell type files into a single file and attach lineage info

In [20]:
def concat_MACSiq(work_dir,preprocessed_dir=preprocessed_dir):
    files = [x for x in work_dir.iterdir() if x.is_file() and x.suffix == '.csv' and x.name != '1 all_info.csv']
    output_dir = Path(preprocessed_dir/work_dir.name)
    if os.path.exists(output_dir):
        pass 
    else:
        output_dir.mkdir(parents=True, exist_ok=True)
    output_file = Path(output_dir / 'macsiq_phenotyping.csv')
    if os.path.exists(output_file):
        print("File exists")
        return pd.read_csv(output_file)
    else:
        pass
    data=[]
    for f in files:
        df=pd.read_csv(f)
        symbol='@'+df['Cell Id'][0].split('@')[1]
        df.insert(1,'cell_type',df.shape[0]*[f.name[0:-4]])
        df.insert(1,'cell_id',''.join(df['Cell Id'].values).split(symbol)[0:-1])
        data.append(df)
    
    df=pd.concat(data,ignore_index=True)
    df["cell_id"] = pd.to_numeric(df["cell_id"])
    df.sort_values(by=['cell_id'],inplace=True,ignore_index=True)
    df_macsiq=df.loc[:,['cell_id','cell_type']]
    df_macsiq['cell_type'] = df_macsiq['cell_type'].str.replace(r'^\d+\s+', '', regex=True)

    level=[]
    source=[]
    cell_type_label=[]
    for c in df_macsiq.cell_type.values:
        info=lineage_info[c]
        level.append(info[0])
        source.append(info[1])
        cell_type_label.append(info[2])

    df_lineage=pd.DataFrame({'level':level,'parent_cell':source,'type_no': cell_type_label })
    df_macsiq=pd.concat([df_macsiq,df_lineage],ignore_index=False,axis=1)

    df_macsiq.to_csv(output_file,index=False)

    return df_macsiq

for work_dir in roi_list:
    concat_MACSiq(work_dir, preprocessed_dir)

File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists
File exists


## 3) Filter out unwanted values: A) phenotyping depths/levels and B) repeated labels such that only the highest phenotyping level/depth remains

In [21]:
def filter_MACSiq(work_dir, preprocessed_dir=preprocessed_dir):
    output_dir = Path(preprocessed_dir/work_dir.name)
    input_file = Path(output_dir / 'macsiq_phenotyping.csv')
    output_file = Path(output_dir/'final_phenotyping_labels.csv')
    if os.path.exists(output_file):
        print(f"{output_file.stem} already exists")
        return pd.read_csv(output_file)
    else:
        pass
    print(f"Using {input_file.stem} to create {output_file.stem}")
    df_macsiq = pd.read_csv(input_file)
    df_macsiq_filt = df_macsiq.loc[df_macsiq['level']<=pheno_depth]
    repeated_labels=[cellID for cellID, rep in (df_macsiq_filt['cell_id'].value_counts()>1).items() if rep==True]
    repeated_labels.sort()
    remove_indices=[]
    for cellID in repeated_labels:
        cell_lineage=df_macsiq_filt.loc[df_macsiq_filt.cell_id==cellID].level
        index=np.argmax(cell_lineage.values)
        final_type_index=cell_lineage.index.values[index]
        remove_elements=np.setdiff1d(cell_lineage.index.values,[final_type_index]).tolist()
        if remove_elements:
            remove_indices.extend(remove_elements)
    df_macsiq_filt=df_macsiq_filt.drop(index=remove_indices)
    df_macsiq_filt.to_csv(output_file,index=False)
    return df_macsiq_filt

for work_dir in roi_list:
    filter_MACSiq(work_dir, preprocessed_dir)

final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists
final_phenotyping_labels already exists


# Preparing the expression matrix and the metadata

## 4) Extract each roi's cells metadata

In [22]:
def meta_MACSiq(work_dir, preprocessed_dir=preprocessed_dir):
    output_dir = Path(preprocessed_dir/work_dir.name)
    output_file = Path(output_dir / 'metadata.csv')
    if os.path.exists(output_file):
        print(f"{output_file.stem} already exists")
        return pd.read_csv(output_file)
    else:
        pass
    print(f"Creating metadata file {output_file.stem}") 
    #load the data with the metadata columns only (79 columns)
    meta = pd.read_csv(work_dir / '1 all_info.csv', usecols=range(1,79))
    #add cell_id column
    meta.insert(0, "cell_id", range(1, 1 + len(meta)))
    #loading the phenotyping data
    df_macsiq_filt = pd.read_csv(output_dir/'final_phenotyping_labels.csv')
    #merge the metadata with the phenotyping data
    meta_joined = df_macsiq_filt.merge(meta, how="left")
    #export the metadata file
    meta_joined.to_csv(output_file,index=False)
    return meta_joined

for work_dir in roi_list:
    meta_MACSiq(work_dir, preprocessed_dir)

metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists
metadata already exists


## 5) Prepare the antibody panel data

In [7]:
#Define a little function for Run92
def split_last_char(s):
    return s[:-1] + ' ' + s[-1] if s[-1].isdigit() else s
#Main function now
#Read the antibodies list
def fix_antibody(panel):
    #check which panel is being used
    if panel.name == "Run92_Ab_panel.xlsx":
        ab = pd.read_excel(panel, sheet_name='IO Cryo Panel')
    else:
        ab = pd.read_excel(panel, sheet_name=["IO core panel", "Add-on Panel"])
        ab = pd.concat(ab, ignore_index=True)
    #Create the antibdoy column that is similar to how the exp_data is formatted
    ab["exp_col"] = ab["ab_clone"].str.replace("__", " ").str.replace("_", " ")
    #add 1 at the end of duplicates
    ab["exp_col"][ab["exp_col"].duplicated(keep="first")] = ab["exp_col"][ab["exp_col"].duplicated(keep="first")] + " 1"

    #Let's create standardized antibody names
    ab["antibody_name"] = ab.apply(lambda x: x["ab_clone"].split(f'{x["clone"]}')[0], axis=1)
    ab["antibody_name"] = ab["antibody_name"].str.replace("__", "_").str.replace("_", "").str.replace("-", "").str.replace(" ", "").sort_values()
    #remove antibodies with low staining quality
    # ab = ab[ab["staining quality"] != "LOW"]
    #add 1 at the end of antibody name if it is duplicated
    ab["antibody_name"][ab["antibody_name"].duplicated(keep="first")] = ab["antibody_name"][ab["antibody_name"].duplicated(keep="first")] + " 1"
    ab["antibody_name"] = ab["antibody_name"].str.replace("Arginase12", "Arginase1")
    ab["antibody_name"] = ab["antibody_name"].str.replace("CollagenIV2", "CollagenIV")
    ab["antibody_name"] = ab["antibody_name"].str.replace("S1001", "S100")
    ab["antibody_name"] = ab["antibody_name"].str.upper()
    ab["exp_col"] = ab["ab_clone"].str.replace("__", " ").str.replace("_", " ")
    ab["exp_col"][ab["exp_col"].duplicated(keep="first")] = ab["exp_col"][ab["exp_col"].duplicated(keep="first")] + " 1"
    #sort them by exp col
    ab = ab.loc[natsorted(ab.index, key=lambda i: ab.loc[i, 'exp_col'])].reset_index(drop=True)
    #special fix for Run92 again
    if panel.name == "Run92_Ab_panel.xlsx":
        ab["antibody_name"] = ab["antibody_name"].apply(split_last_char)
    #export it to csv
    ab.to_csv(metadata_dir/f"{panel.stem}.csv", index=False)
    return ab

ab_list = []
for panel in ab_panels:
    ab = fix_antibody(panel)
    ab_name = np.array(natsorted(ab["antibody_name"]))
    ab_list.append(ab_name)
    # ab_intersection = set(ab_list[0]).intersection(*ab_list)

/var/folders/zj/m3tnl76s7d5894m0ndctg1kh0000gn/T/ipykernel_40572/2288292176.py:16: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  ab["exp_col"][ab["exp_col"].duplicated(keep="first")] = ab["exp_col"][ab["exp_col"].duplicated(keep="first")] + 

## 6) Extract expression data 

In [21]:
def exp_MACSiq(work_dir, preprocessed_dir=preprocessed_dir):
    output_dir = Path(preprocessed_dir/work_dir.name)
    output_file = Path(output_dir / 'exp.csv')
    run = work_dir.name.split('_')[0]
    if os.path.exists(output_file):
        print(f"{output_file.stem} already exists")
        return pd.read_csv(output_file)
    else:
        pass
    print("Creating expression file", output_file)
    #load the necessary antibody list
    ab_exp = pd.read_csv(metadata_dir/f"{run}_Ab_panel.csv")
    #load the expression data
    all_columns = pd.read_csv(work_dir / '1 all_info.csv', nrows=0).columns
    selected_columns = all_columns[all_columns.str.contains("Cell Exp")]
    #Columns must not be DAPI, APC, PE 
    selected_columns = [col for col in selected_columns if not any(x in col for x in ['DAPI', 'APC', 'PE', 'FITC'])]
    exp = pd.read_csv(work_dir / '1 all_info.csv', usecols=selected_columns)
    #remove ' Cell Exp' from column names
    exp.columns = exp.columns.str.replace(' Cell Exp', "")
    #Filter out the columns for only ab_exp columns
    available_cols = [col for col in exp.columns if col in ab_exp["exp_col"].values]
    ab_exp = ab_exp[ab_exp["exp_col"].isin(available_cols)]
    exp = exp.loc[:,available_cols]
    #reorder ab_exp
    ab_exp = ab_exp.sort_values(by="exp_col", key=lambda x: x.map({col: i for i, col in enumerate(available_cols)}))
    #Fix the names of the antibodies
    exp.columns = ab_exp["antibody_name"].values
    #Filter out the columns with low staining quality
    low_quality = ab_exp[ab_exp["staining quality"] == "LOW"]["antibody_name"]
    exp = exp.drop(columns=low_quality, errors='ignore')
    #drop low quality from ab_exp
    ab_exp = ab_exp[~ab_exp["antibody_name"].isin(low_quality)]
    #add cell_id column
    exp.insert(0, 'cell_id', range(1, 1 + len(exp)))
    #Only keep rows with a cell type (Remove trash cells)
    meta = pd.read_csv(output_dir / 'metadata.csv')
    cell_ids = meta['cell_id']
    exp = exp[exp['cell_id'].isin(cell_ids)]
    #resort the columns
    exp = exp[['cell_id'] + natsorted(exp.columns[1:])]
    #export the expression data
    exp.to_csv(output_file,index=False)
    return exp
 
for work_dir in roi_list:
    exp_MACSiq(work_dir, preprocessed_dir)

Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI11/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI16/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI7/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run145_ROI3/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI19/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI1/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI8/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI17/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run145_ROI10/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run145_ROI17/exp.csv
Creating expression file /Users/ja

## 7) Extract patients metadata and export it to the new format

In [22]:
patients_metadata = pd.read_excel(metadata_dir / 'patient_metadata.xlsx')
patients_metadata["outcome"] = patients_metadata["outcome"].str.extract('(\d+)').astype(int)
#remove excluded rois
patients_metadata = patients_metadata.loc[~patients_metadata["rois"].isna()]
#add a new column if the  "outcome" column is greater than 9
patients_metadata["outcome_group"] = np.where(patients_metadata["outcome"] > 6, "long_term survival", "short_term survival")
patients_metadata["rois"] = patients_metadata["rois"].astype(str)
patients_metadata['rois'] = patients_metadata['rois'].str.split(r'\+ ')
patients_metadata = patients_metadata.explode('rois', ignore_index=True)
patients_metadata['rois'] = patients_metadata['rois'].str.strip()
for i in range(len(patients_metadata)):
    if patients_metadata['rois'][i].find('-') != -1:
        new_patient = patients_metadata['rois'][i].split('-')
        new_patient = np.array(range(int(new_patient[0]), int(new_patient[1])+1))
        patients_metadata['rois'][i] = new_patient
patients_metadata = patients_metadata.explode('rois', ignore_index=True)
patients_metadata["rois"] = patients_metadata["rois"].astype(str)
patients_metadata["exp_name"] = "Run" +  patients_metadata['run'].astype(int).astype(str) + '_ROI' + patients_metadata['rois'].astype(str)
patients_metadata.to_csv(metadata_dir / 'new_patients_metadata.csv', index=False)

/var/folders/zj/m3tnl76s7d5894m0ndctg1kh0000gn/T/ipykernel_40572/4150435120.py:15: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  patients_metadata['rois'][i] = new_patient
/var/folders/zj/m3tnl76s7d5894m0ndctg1kh0000gn/T/ipykernel_40572/4150

## 8) Remove the trash cells

In [23]:
processed_roi_list = [roi for roi in preprocessed_dir.iterdir() if roi.is_dir() and roi.name != '.DS_Store']
for roi in processed_roi_list:
    exp = pd.read_csv(roi / 'exp.csv')
    meta = pd.read_csv(roi / 'metadata.csv')
    if "Trash" not in meta["cell_type"].unique():
        print("No trash cells in metadata")
        continue
    else:
        pass
    print("Removing trash cells")
    meta = meta.loc[meta["cell_type"] != "Trash"]
    exp = exp[exp["cell_id"].isin(meta.cell_id)]
    #export csvs
    exp.to_csv(roi / 'exp.csv', index=False)
    meta.to_csv(roi / 'metadata.csv', index=False)

No trash cells in metadata
No trash cells in metadata
Removing trash cells
Removing trash cells
No trash cells in metadata
No trash cells in metadata
Removing trash cells
Removing trash cells
Removing trash cells
Removing trash cells
Removing trash cells
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
Removing trash cells
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
Removing trash cells
No trash cells in metadata
Removing trash cells
Removing trash cells
No trash cells in metadata
No trash cells in metadata


## 9) Quick QC : Extract the names of all cell types

In [24]:
# Create a list of all cell types
all_cell_types = []  
for i, roi in enumerate(processed_roi_list):
    meta = pd.read_csv(roi / 'metadata.csv')
    all_cell_types.extend(meta["cell_type"].unique())
# Create a set of unique cell types
all_cell_types = sorted(set(all_cell_types))   
# Create a DataFrame with all cell types and initialize counts to zero
meta = pd.read_csv(processed_roi_list[0] / 'metadata.csv')
meta_counts = meta["cell_type"].value_counts()
for cell_type in all_cell_types:
        if cell_type not in meta_counts:
            meta_counts[cell_type] = 0
sorted_order = meta_counts.index  # Extract the index order from a reference DataFrame
all_cell_types = list(sorted_order)
all_cell_types

['Tumor_cells',
 'Actin+_cells',
 'NFC',
 'Endothelial_cells',
 'CD4+_T_cells',
 'M1_macrophages',
 'DCs',
 'M1_M2_macrophages',
 'Mast_cells',
 'M2_macrophages',
 'CD8+_T_cells',
 'Treg_T_cells',
 'Plasma_cells',
 'Lymphatic_endothelial_cells',
 'MDSCs',
 'B_cells',
 'Granulocytes',
 'Epithelial_cells',
 'NK_cells']

## 10) Quick QC : Checking  the number of antibodies in all samples and runs

In [26]:
exp = pd.read_csv(processed_roi_list[0]/'exp.csv')
exp.iloc[:,:15]

,cell_id,ACTIN,ARGINASE1,BCL2,BETAACTIN,BETACATENIN,CALPONIN,CD1C,CD3,CD4,CD8A,CD11B,CD11C,CD14,CD15
0,1,138.283493,78.628265,371.949341,76.404411,509.568848,2554.586426,253.948532,799.354553,293.530182,1470.500000,214.027771,476.651154,90.171944,199.847229
1,2,146.363861,81.042168,388.899994,79.098793,482.771088,2611.408447,223.055420,803.173523,272.330109,1421.838501,191.284332,481.163849,84.333733,207.315659
2,3,229.803131,77.978745,372.837799,77.362419,449.423950,2410.932861,194.076065,762.546997,243.024612,1235.976562,162.784119,442.539154,73.534676,211.564880
3,4,190.582901,113.010361,415.054413,89.981865,460.489624,2824.538818,235.310883,838.246094,295.108795,1358.834229,186.375641,567.427490,86.531090,204.733154
4,5,184.437149,109.067780,402.783356,88.893608,492.359924,2636.151123,239.076355,795.616455,304.922363,1365.983643,202.195190,500.087097,93.113686,204.060913
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18777,18791,160.029770,303.144470,497.733795,140.162003,837.062195,3564.399414,355.679504,1071.901001,466.112946,1898.638306,317.274078,717.819641,126.228546,225.048157
18778,18792,257.980042,247.922394,552.616394,135.784927,633.727295,4141.902344,294.073181,1187.481201,451.625275,2303.702881,305.702881,737.855896,137.388031,281.815979
18779,18793,152.509232,293.121552,535.039978,141.230774,793.666138,3461.255371,348.166168,1097.180054,447.033844,2036.249268,319.918457,750.049255,141.318466,246.310776
18780,18794,1109.359253,157.215302,310.596863,117.788834,245.458328,1505.644165,134.139725,486.505249,179.311050,769.273560,169.277328,333.158844,45.256435,131.481781


In [27]:
for roi in processed_roi_list:
    exp = pd.read_csv(roi / 'exp.csv')
    print (roi.name)
    print(len(exp.columns))

Run141_ROI11
86
Run141_ROI16
86
Run141_ROI7
86
Run145_ROI3
75
Run141_ROI19
86
Run141_ROI1
86
Run141_ROI8
86
Run141_ROI17
86
Run145_ROI10
75
Run145_ROI17
71
Run145_ROI11
75
Run103_ROI5
61
Run92_ROI5
53
Run141_ROI15
86
Run145_ROI1
75
Run141_ROI4
86
Run141_ROI23
86
Run141_ROI3
86
Run145_ROI9
75
Run141_ROI14
87
Run145_ROI14
75
Run145_ROI13
75
Run103_ROI6
61
Run92_ROI6
53


In [28]:
#only rois in processed_roi_list with 145 in their name
rois_145 = [roi for roi in processed_roi_list if '145' in roi.name]

for roi in rois_145:
    #Only read column names
    exp = pd.read_csv(roi / 'exp.csv', nrows=0)
    print(roi.name)
    print(len(exp.columns))

test1 = pd.read_csv(rois_145[2] / 'exp.csv')
test2 = pd.read_csv(rois_145[0] / 'exp.csv')
set(test2.columns) - set(test1.columns)

Run145_ROI3
75
Run145_ROI10
75
Run145_ROI17
71
Run145_ROI11
75
Run145_ROI1
75
Run145_ROI9
75
Run145_ROI14
75
Run145_ROI13
75


{'BCL2', 'CD1C', 'CD209', 'CD279 1'}

In [29]:
#only rois in processed_roi_list with 141 in their name
rois_141 = [roi for roi in processed_roi_list if '141' in roi.name]

for roi in rois_141:
    #Only read column names
    exp = pd.read_csv(roi / 'exp.csv', nrows=0)
    print(roi.name)
    print(len(exp.columns))

missing_roi = [roi for roi in rois_141 if '141_ROI14' in roi.name]
random_roi = [roi for roi in rois_141 if '141_ROI3' in roi.name]
test1 = pd.read_csv(missing_roi[0] / 'exp.csv')
test2 = pd.read_csv(random_roi[0] / 'exp.csv')
set(test1.columns) - set(test2.columns)

Run141_ROI11
86
Run141_ROI16
86
Run141_ROI7
86
Run141_ROI19
86
Run141_ROI1
86
Run141_ROI8
86
Run141_ROI17
86
Run141_ROI15
86
Run141_ROI4
86
Run141_ROI23
86
Run141_ROI3
86
Run141_ROI14
87


{'CD279 1'}

In [30]:
for a in roi_list[0:10]:
    batikh = pd.read_csv(a / '1 all_info.csv', nrows=0).columns
    #search batikh for columns with Bcl2
    batikh = [col for col in batikh if 'Cell Exp' in col]
    batikh = [col for col in batikh if 'ROI' in col]
    print(a)
    print(batikh)

    #Bcl 2 CD209 CD279
    # NUT C52B1 #CD279

/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run141_ROI11
[]
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run141_ROI16
[]
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run141_ROI7
[]
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run145_ROI3
[]
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run141_ROI19
[]
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run141_ROI1
[]
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run141_ROI8
[]
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run141_ROI17
[]
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run145_ROI10
[]
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run145_ROI17
['Bcl 2 REA872 ROI17 Cell Exp', 'CD209 REAL690 ROI17 Cell Exp', 'CD279 REA1165 ROI17 Cell Exp', 'DAPI ROI17 C0 Cell Exp', 'DAPI ROI17 C1 Cell Exp']


In [31]:
for a in roi_list[10:20]:
    batikh = pd.read_csv(a / '1 all_info.csv', nrows=0).columns
    #search batikh for columns with Bcl2
    batikh = [col for col in batikh if 'Cell Exp' in col]
    batikh = [col for col in batikh if 'CD279' in col]
    print(a)
    print(batikh)

    #Bcl 2 CD209 CD279
    # NUT C52B1 #CD279

/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run145_ROI11
['CD279 REA1165 Cell Exp', 'CD279 REA1165 1 Cell Exp']
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run103_ROI5
['CD279 REA1165 Cell Exp']
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run92_ROI5
['CD279 1 Cell Exp']
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run141_ROI15
['CD279 REA1165 Cell Exp']
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run145_ROI1
['CD279 REA1165 Cell Exp', 'CD279 REA1165 1 Cell Exp']
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run141_ROI4
['CD279 REA1165 Cell Exp', 'CD279 REA1165 ROI4 Cell Exp']
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run141_ROI23
['CD279 REA1165 Cell Exp']
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run141_ROI3
['CD279 REA1165 Cell Exp']
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run145_ROI9
['CD279 REA1165 Cell Exp', 'CD279 REA1165 1 Cell Exp']
/Users/jawadalaaedeen/Desktop/PhD/NMC/raw/Run141_ROI14
['CD279 REA1165 Cell Exp', 'CD279 REA1165 1 Cell Exp']


## 11) Extract the missing columns from ROI17 to fix it

In [32]:
Run145_ROI17 = [roi for roi in roi_list if 'Run145_ROI17' in roi.name]
Run145_ROI17
Run145_ROI17_processed = [roi for roi in processed_roi_list if 'Run145_ROI17' in roi.name]
Run145_ROI17_processed

[PosixPath('/Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run145_ROI17')]

In [33]:
#9 and #15 
#from 9 (Run145_ROI17)
missing_exp = pd.read_csv(Run145_ROI17[0] / '1 all_info.csv', nrows=0).columns
#search batikh for columns with Bcl2
missing_exp = [col for col in missing_exp if 'Cell Exp' in col]
missing_exp = [col for col in missing_exp if 'ROI' in col]
#take out the DAPI, APC, PE
missing_exp = [col for col in missing_exp if not any(x in col for x in ['DAPI', 'APC', 'PE'])]     
missing_exp
#extract those columns
missing_exp = pd.read_csv(Run145_ROI17[0] / '1 all_info.csv', usecols=missing_exp)
missing_exp.columns = ["BCL2", "CD209", "CD279 1"]
#create cell_id column
missing_exp.insert(0, 'cell_id', range(1, 1 + len(missing_exp)))
#remove the trash cells
meta = pd.read_csv(Run145_ROI17_processed[0] / 'metadata.csv')
meta = meta.loc[meta["cell_type"] != "Trash"]
missing_exp = missing_exp[missing_exp["cell_id"].isin(meta.cell_id)].reset_index(drop=True)
#drop cell_id column
missing_exp = missing_exp.drop(columns=["cell_id"])
# #add it to the exp data
exp = pd.read_csv(Run145_ROI17_processed[0] / 'exp.csv')
#cbind the missing columns
exp = pd.concat([exp, missing_exp], axis=1)
#reorder the columns with natsorted except cell_id at first
exp = exp[['cell_id'] + natsorted(exp.columns[1:])]
#export the exp data and rewrite it for this one
exp.to_csv(Run145_ROI17_processed[0] / 'exp.csv', index=False)

## Final step is to remove the CD1C for RUN145 all except 1 and remove CD279 1 from one ROI in Run141


### Run145

In [35]:
exp = pd.read_csv(rois_145[2]/"exp.csv")
exp_final_columns = exp.columns
for roi in rois_145:
    exp = pd.read_csv(roi/"exp.csv")
    exp = exp[exp_final_columns]
    exp.to_csv(roi/"exp.csv", index=False)

### Run141

In [36]:
exp = pd.read_csv(rois_141[0]/"exp.csv")
exp_final_columns = exp.columns
exp = pd.read_csv(rois_141[11]/"exp.csv")
exp = exp[exp_final_columns]
exp.to_csv(rois_141[11]/"exp.csv", index=False)